# 경제 분석 및 예측과 데이터 지능 실습2.5 (sktime 버전)

Kaggle **Store Sales - Time Series Forecasting** 데이터를 대상으로, sktime로 다음을 수행합니다.

- (1) EDA (요약)
- (2) 단일 시계열 예측 (exogenous 포함)
- (3) Appendix: Fourier features (계절성 특징)

## 데이터 위치
이 노트북은 아래 폴더에 Kaggle 데이터를 기대합니다.

- `../datasets/kaggle_store_sales/train.csv`
- `../datasets/kaggle_store_sales/test.csv`
- `../datasets/kaggle_store_sales/stores.csv`
- `../datasets/kaggle_store_sales/transactions.csv`
- `../datasets/kaggle_store_sales/oil.csv`
- `../datasets/kaggle_store_sales/holidays_events.csv`

데이터가 없으면 **에러 없이** 작은 synthetic 예제로 모델 파트만 실행합니다.

In [ ]:
# --- Import hygiene (repo contains ./sktime source checkout) ---
# Running from repo root can shadow `import sktime` via ./sktime (namespace).
# Force using the actual package at ./sktime/sktime by adjusting sys.path.
import os
import sys

repo_root = os.path.abspath(os.getcwd())
local_sktime_src = os.path.join(repo_root, "sktime")
if os.path.exists(os.path.join(local_sktime_src, "sktime", "__init__.py")):
    sys.path = [p for p in sys.path if p not in ("", repo_root)]
    sys.path.insert(0, local_sktime_src)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.model_selection import temporal_train_test_split
from sktime.performance_metrics.forecasting import mean_absolute_percentage_error
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.compose import ForecastingPipeline, RecursiveTabularRegressionForecaster
from sktime.transformations.series.fourier import FourierFeatures

plt.rcParams["figure.figsize"] = (11, 4)

## 1) 데이터 로드

In [ ]:
base = "../datasets/kaggle_store_sales"
paths = {
    "train": os.path.join(base, "train.csv"),
    "test": os.path.join(base, "test.csv"),
    "stores": os.path.join(base, "stores.csv"),
    "transactions": os.path.join(base, "transactions.csv"),
    "oil": os.path.join(base, "oil.csv"),
    "holidays": os.path.join(base, "holidays_events.csv"),
}

has_kaggle = all(os.path.exists(p) for p in paths.values())
has_kaggle

In [ ]:
if has_kaggle:
    train = pd.read_csv(paths["train"], parse_dates=["date"])
    stores = pd.read_csv(paths["stores"])
    transactions = pd.read_csv(paths["transactions"], parse_dates=["date"]).sort_values(["store_nbr", "date"])
    oil = pd.read_csv(paths["oil"], parse_dates=["date"])
    holidays = pd.read_csv(paths["holidays"], parse_dates=["date"])

    display(train.head())
else:
    # Synthetic fallback that still exercises sktime forecasting with X.
    idx = pd.date_range("2017-01-01", periods=365 * 3, freq="D")
    rng = np.random.default_rng(42)
    y = (
        200
        + 0.02 * np.arange(len(idx))
        + 10 * np.sin(2 * np.pi * np.arange(len(idx)) / 7)
        + 30 * np.sin(2 * np.pi * np.arange(len(idx)) / 365.25)
        + rng.normal(0, 5, size=len(idx))
    )
    y = pd.Series(y, index=idx, name="sales")
    X = pd.DataFrame(
        {
            "oil": 50 + 3 * np.sin(2 * np.pi * np.arange(len(idx)) / 365.25) + rng.normal(0, 0.8, size=len(idx)),
            "onpromotion": (rng.random(len(idx)) < 0.1).astype(int),
        },
        index=idx,
    )
    print("Kaggle data not found. Using synthetic series for modeling demo.")
    display(y.head())

## 2) 단일 시계열 예측 (exogenous 포함)

NeuralProphet의 `lagged/future regressors` 개념은 sktime에서 **(y, X)** 로 자연스럽게 처리합니다.

In [ ]:
if has_kaggle:
    # One store/family to keep runtime reasonable
    one = train[(train["store_nbr"] == 1) & (train["family"] == "GROCERY I")].copy().sort_values("date")
    y = one.set_index("date")["sales"].asfreq("D").astype(float).interpolate(limit_direction="both")

    oil_ = oil.set_index("date")["dcoilwtico"].asfreq("D").interpolate(limit_direction="both")
    promo_ = one.set_index("date")["onpromotion"].asfreq("D").fillna(0)
    X = pd.DataFrame({"oil": oil_, "onpromotion": promo_}, index=y.index).interpolate(limit_direction="both")

y_train, y_test, X_train, X_test = temporal_train_test_split(y, X, test_size=90)
fh = ForecastingHorizon(y_test.index, is_relative=False)

baseline = NaiveForecaster(strategy="last", sp=7)
baseline.fit(y_train)
yhat_base = baseline.predict(fh)

model = ForecastingPipeline(
    steps=[
        ("fourier", FourierFeatures(sp_list=[7, 365.25], fourier_terms_list=[3, 5])),
        (
            "reg",
            RecursiveTabularRegressionForecaster(
                estimator=RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
                window_length=28,
            ),
        ),
    ]
)

model.fit(y_train, X=X_train)
yhat = model.predict(fh, X=X_test)

ax = y_train.iloc[-180:].plot(label="train", alpha=0.7)
y_test.plot(ax=ax, label="test", alpha=0.9)
yhat_base.plot(ax=ax, label="SeasonalNaive(sp=7)")
yhat.plot(ax=ax, label="Fourier+RF(recursive)")
ax.legend()
ax.set_title("Single-series forecasting with exogenous X")
plt.show()

print("MAPE baseline:", float(mean_absolute_percentage_error(y_test, yhat_base)))
print("MAPE model:", float(mean_absolute_percentage_error(y_test, yhat)))

## APPENDIX: Fourier features

Fourier 특징을 만들어 보면 아래와 같습니다.

In [ ]:
ff = FourierFeatures(sp_list=[7, 365.25], fourier_terms_list=[1, 1])
X_fourier = ff.fit_transform(y_train)
X_fourier.head()